### Middleware


Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

- Tracking agent behaviour with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"]

#### Summarization Middleware


Automatically summarize coversation history when approaching token limits, perserving recent messages while compressing older context. Summarization is useful for the following:

- Long running conversations that exceed context windows
- Multi turn dialogues with extensive history
- Applications where preserving full conversation context matters.


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage

Agent having middleware that depends on messages.length


In [ ]:
agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    tools=[],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:llama-3.3-70b-versatile",
            trigger=("messages", 10),  # It will trigger after messages.length >= 10
            keep=("messages", 4),      # Persist only latest 4 messages
        ),
    ],
)

In [ ]:
questions = [
    "Provide solutions to upcoming problems in under 100 words." "What is HTML?",
    "Is there any alternative to HTML?",
    "What are the key concepts in HTML?",
    "Are HTML and HTTP different?",
    "What's the difference between HTTP and HTTPs?",
]

for question in questions:
    response = agent.invoke(
        {"messages": [HumanMessage(content=question)]},
        config={"configurable": {"thread_id": "test-1"}},
    )
    print(f"Messages: {response}")
    print(f"Length of messages: {len(response['messages'])}")

Agent having middleware that depends on token size


In [ ]:
from langchain_core.tools import tool

@tool
def search_hotels(city: str):
    """Search for hotels - returns long response to use more tokens"""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $210/night, business center
    3. Stay Here - 4 star, $170/night, free wifi"""


agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:llama-3.3-70b-versatile",
            trigger=("tokens", 550),
            keep=("tokens", 250),
        )
    ],
)


def count_tokens(messages):
    total_chars = sum(len(str(message.content)) for message in messages)
    return total_chars

In [ ]:
cities = ["Paris", "London", "Tokyo", "New York", "Dubai"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config={"configurable": {"thread_id": "test_1"}},
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: {tokens} tokens used, {len(response['messages'])} messages")
    print(f"{response['messages']}")

Agent having middleware that depends on fraction of models overall context size


In [ ]:
@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"


agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:llama-3.3-70b-versatile",
            trigger=("fraction", 0.005),  # 0.5% = ~655 tokens
            keep=("fraction", 0.002),  # 0.2% = ~262 tokens
        ),
    ],
)


# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4  # 4 char ~ 1 token


# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config={"configurable": {"thread_id": "test-1"}},
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 131072
    print(
        f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs"
    )
    print(response["messages"])

#### Human In the Loop Middleware


Pause the agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

- High stakes operations requiring human approval (eg. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long running conversations where human feedback guides the agent.


In [ ]:
def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its email id"""
    return f"Email content for ID: {email_id}"


def send_email_tool(recipent: str, subject: str, body: str) -> str:
    """Mock function to send an email"""
    return f"Email sent to {recipent} with subject '{subject}'"

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {"allowed_decisions": ["respond", "edit", "reject"]},
                "send_email_tool": False,
            }
        )
    ],
)

In [ ]:
result = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Send email to john@test.com with subject 'Hello John' and body 'How are you John?"
            )
        ]
    },
    config={"configurable": {"thread_id": "test_1"}},
)

result

In [ ]:
from langgraph.types import Command

if "__interrupt__" in result:
    print("Paused for approval...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config={"configurable": {"thread_id": "test-1"}},
    )

    print(f"Result: {result['messages'][-1].content}")

In [ ]:
result

Reject

In [ ]:
if "__interrupt__" in result:
    print("Paused for Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config={"configurable": {"thread_id": "test-1"}}
    )
    
    print(f"Result: {result['messages'][-1].content}")

In [ ]:
result

Editing

In [ ]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("Paused for Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config={"configurable": {"thread_id": "test-1"}}
    )
    
    print(f"Result: {result['messages'][-1].content}")

In [ ]:
result